# Problem: Implement an RNN from Scratch

### Problem Statement
Implement a vanilla (Elman) **recurrent neural network** with raw tensor operations instead of `nn.RNN`. Train it to predict the next value of a sine wave from the previous 10 values.

### The recurrence
For an input sequence $x_1, \dots, x_T$ with $x_t \in \mathbb{R}^{d_{in}}$:

$$h_0 = 0, \qquad h_t = \tanh\big(x_t W_{ih} + h_{t-1} W_{hh} + b_h\big), \qquad \hat{y} = h_T W_{out} + b_{out}$$

The same weights are used at every time step (weight sharing). The hidden state $h_t$ is the only thing passed from one step to the next. After the last step, $h_T$ summarizes the whole sequence.

### Requirements
1. **`RNNModel(input_dim=1, hidden_dim=50, output_dim=1)`**:
   - Create `W_ih` with shape `(input_dim, hidden_dim)`, `W_hh` with shape `(hidden_dim, hidden_dim)` and `b_h` with shape `(hidden_dim,)` as `nn.Parameter`s, plus an output layer `fc = nn.Linear(hidden_dim, output_dim)`.
   - `forward(x)` takes `x` of shape `(batch, seq_len, input_dim)`. It starts `h` at zeros, shape `(batch, hidden_dim)`, created fresh on every call on `x`'s device and with `x`'s dtype. It loops over time and returns `fc(h_T)`, shape `(batch, output_dim)`.
   - It must work for any batch size and any sequence length.
2. **Do not use `nn.RNN` or `nn.RNNCell` inside the model.** A test cell compares your forward pass with `nn.RNN` after copying your weights into it.

### Notes an interviewer may probe
- `nn.RNN` computes $\tanh(x_t W_{ih}^\top + b_{ih} + h_{t-1} W_{hh}^\top + b_{hh})$. It stores its weights transposed relative to ours, as `(hidden, input)`, and it has two bias vectors whose sum plays the role of $b_h$.
- The input projection $x_t W_{ih}$ does not depend on $h$. It can be computed for all time steps in one matmul before the loop, which is what cuDNN does. Only $h_{t-1} W_{hh}$ is truly sequential.
- Backpropagation through time is just autograd through the Python loop. With our row-vector convention, $\partial L/\partial h_{t-1} = (\partial L/\partial h_t)\,\operatorname{diag}(1 - h_t^2)\, W_{hh}^\top$, so the gradient is multiplied by one such factor per step, and over long sequences it tends to vanish (or explode if $W_{hh}$ has a large spectral radius). That is the motivation for LSTMs and GRUs and for gradient clipping.

<details>
  <summary>💡 Hint</summary>
  Keep one tensor <code>h</code> of shape <code>(batch, hidden_dim)</code> that starts at zeros (<code>x.new_zeros(...)</code> matches <code>x</code>'s device and dtype). Overwrite it once per time step using the recurrence above, reading the step's input as <code>x[:, t]</code>. Apply <code>fc</code> only after the loop.
  <br>
  Initialize the weights small (e.g. <code>torch.randn(...) * 0.1</code>) so that <code>tanh</code> does not saturate at the start.
</details>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
# Define the RNN Model
class RNNModel(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=50, output_dim=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        # Shared by every time step. Scale 0.1 keeps tanh out of saturation at the start
        # (W_hh's spectral radius is about 0.1 * sqrt(hidden_dim) ~ 0.7 < 1).
        self.W_ih = nn.Parameter(torch.randn(input_dim, hidden_dim) * 0.1)
        self.W_hh = nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.1)
        self.b_h = nn.Parameter(torch.zeros(hidden_dim))
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        # The input projection does not depend on h: do it for all steps in one matmul
        x_proj = x @ self.W_ih + self.b_h             # (batch, seq_len, hidden_dim)
        h = x.new_zeros(batch_size, self.hidden_dim)  # h_0 = 0, fresh for every batch
        for t in range(seq_len):
            h = torch.tanh(x_proj[:, t] + h @ self.W_hh)
        return self.fc(h)                             # h_T summarizes the whole sequence

In [ ]:
# Check: same math as PyTorch's nn.RNN once the weights are copied over
torch.manual_seed(0)
ours = RNNModel(input_dim=3, hidden_dim=8, output_dim=2)
ref = nn.RNN(input_size=3, hidden_size=8, batch_first=True)  # tanh nonlinearity by default
with torch.no_grad():
    ref.weight_ih_l0.copy_(ours.W_ih.T)  # nn.RNN stores (hidden, input)
    ref.weight_hh_l0.copy_(ours.W_hh.T)
    ref.bias_ih_l0.copy_(ours.b_h)
    ref.bias_hh_l0.zero_()               # nn.RNN's two biases add up to our b_h
    for batch, seq_len in [(4, 7), (1, 1), (2, 25)]:
        x = torch.randn(batch, seq_len, 3)
        _, h_T = ref(x)                  # h_T: (num_layers, batch, hidden)
        out = ours(x)
        assert out.shape == (batch, 2), f"expected {(batch, 2)}, got {tuple(out.shape)}"
        assert torch.allclose(out, ours.fc(h_T[-1]), atol=1e-5), "forward pass does not match nn.RNN"
print("Matches nn.RNN")

In [ ]:
# Generate synthetic sequential data
torch.manual_seed(42)
sequence_length = 10
num_samples = 100

# A sine wave sampled at 100 points over two periods
y = torch.sin(torch.linspace(0, 4 * torch.pi, steps=num_samples).unsqueeze(1))  # (100, 1)


# Prepare data for RNN: each input is a window of 10 values, the target is the value that follows
def create_in_out_sequences(data, seq_length):
    in_seq = []
    out_seq = []
    for i in range(len(data) - seq_length):
        in_seq.append(data[i : i + seq_length])
        out_seq.append(data[i + seq_length])
    return torch.stack(in_seq), torch.stack(out_seq)


X_seq, y_seq = create_in_out_sequences(y, sequence_length)  # (90, 10, 1), (90, 1)

In [ ]:
# Initialize the model, loss function, and optimizer
torch.manual_seed(0)
model = RNNModel()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Train on all 90 windows at once: the model processes a whole batch of sequences in parallel
epochs = 300
for epoch in range(epochs):
    model.train()
    outputs = model(X_seq)            # (90, 1)
    loss = criterion(outputs, y_seq)

    optimizer.zero_grad()
    loss.backward()                   # backpropagation through time
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.6f}")

In [ ]:
# Testing on new data: windows from the continuation of the wave, sampled with the same step
model.eval()
step = 4 * torch.pi / (num_samples - 1)
t_test = 4 * torch.pi + step * torch.arange(1, 61, dtype=torch.float32).unsqueeze(1)
X_test, y_test = create_in_out_sequences(torch.sin(t_test), sequence_length)  # (50, 10, 1), (50, 1)

with torch.no_grad():
    predictions = model(X_test)  # predict the next value of each window

print(f"Test MSE on {len(X_test)} unseen windows: {criterion(predictions, y_test).item():.6f}")
for pred, target in zip(predictions[:5, 0].tolist(), y_test[:5, 0].tolist()):
    print(f"predicted {pred:+.4f}   actual {target:+.4f}")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("rnn", RNNModel=RNNModel)